# Homework 3: Duration prediction model pipeline

##### Abraham Alvarado Padilla

In [12]:
import pandas as pd
import numpy as np
import pickle as pk
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import mlflow 
from mlflow.tracking import MlflowClient
import os

In [7]:
mlflow.set_experiment("Homework_3_First_Pipeline")
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.sklearn.autolog()
os.makedirs("artifacts", exist_ok=True)


2025/06/04 16:25:36 INFO mlflow.tracking.fluent: Experiment with name 'Homework_3_First_Pipeline' does not exist. Creating a new experiment.


In [8]:

#Answer to Question 3: 3,403,766


In [9]:
## Code we put in the pipeline
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    

    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    numerical = ["trip_distance"]
    df[categorical] = df[categorical].astype(str)
    
    return df[categorical + numerical],df["duration"]
df_pipe, y = read_dataframe("yellow_tripdata_2023-03.parquet")
print(df_pipe.shape)
#Answer to Question 4: 3,316,216

(3316216, 3)


In [11]:
def train_model(df_train,y):
    with mlflow.start_run():
        train_dicts = df_train.to_dict(orient="records")
        dv = DictVectorizer()
        X_train = dv.fit_transform(train_dicts)
        y_train = y
        lr = LinearRegression()
        lr.fit(X_train, y_train)
        with open("artifacts/dictvectorizer.pkl", "wb") as f_out:
            pk.dump(dv, f_out)
        mlflow.log_artifact("artifacts/dictvectorizer.pkl", artifact_path = "preprocessors")
train_model(df_pipe,y)
 #The interception i got is 23.8483

2025/06/04 16:29:10 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: Unable to allocate 12.8 GiB for an array with shape (3316216, 519) and data type float64


🏃 View run powerful-vole-661 at: http://127.0.0.1:5000/#/experiments/785480266524284082/runs/aa9c11a0fb0b497eb5552d1c2672e9ed
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/785480266524284082


In [16]:
#Let's register the model
client = MlflowClient()
experiment = client.get_experiment_by_name("Homework_3_First_Pipeline")
model_id = client.search_runs(
    experiment_ids = [experiment.experiment_id],
    filter_string = "metrics.training_root_mean_squared_error > 0",
    order_by = ["metrics.training_root_mean_squared_error ASC"],
    max_results = 1
)
model_id_link = model_id[0].info.run_id
mlflow.register_model(
    model_uri = f"runs:/{model_id_link}/model",
    name = "Best_Linearregression_prediction_model"
)


Successfully registered model 'Best_Linearregression_prediction_model'.
2025/06/04 16:49:37 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Best_Linearregression_prediction_model, version 1
Created version '1' of model 'Best_Linearregression_prediction_model'.


<ModelVersion: aliases=[], creation_timestamp=1749077376834, current_stage='None', description='', last_updated_timestamp=1749077376834, name='Best_Linearregression_prediction_model', run_id='aa9c11a0fb0b497eb5552d1c2672e9ed', run_link='', source='mlflow-artifacts:/785480266524284082/aa9c11a0fb0b497eb5552d1c2672e9ed/artifacts/model', status='READY', status_message=None, tags={}, user_id='', version='1'>